# Pywikibot Sitelink Retrieval

## Retrieve Sitelink Counts and Filter Input QIDs

Input: CSV files containing entity QIDs \
Output: JSON files containing entity QIDs and number of sitelinks for all entities with at least as many sitelinks as input languages.

Input files: https://drive.google.com/drive/folders/1UU4vMOeWISJA-XhuduPgEZhGsZBMllw1 \
Output files: https://drive.google.com/drive/folders/13VTPzeKkonprQzXTfuDrVE2hw6SBqC6C
    

**Steps:** 

Retrieve total number of sitelinks\
Out of those with > 8 sitelinks, select those with Wikipedia pages in all of our languages:

* German (Q188) , Zurich German (Q248682), Austrian German (Q306626)
* French (Q150), Canadian French (Q1450506)
* Italian (Q652)
* Polish (Q809)
* Hindi (Q1568 )
* English (Q1860), American English (Q7976), British English (Q7979), Canadian English(Q44676), Australian English(Q44679)
* Russian (Q7737)
* Chinese (Q7850)
* Arabic (Q13955), Egyptian Arabic (Q29919)

Retrieve the number of sitelinks. 

**Notes:** 

Zurich German does not have a language code: https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all \

At least for Chinese (zh) and German (de) more variants exist than we initially listed. The function is_subset() therefore checks not only for zh and de, but also for all variants of these languages which follow the pattern zh-foo, or de-bar. 

Sitelinks for a few entities in each language could not be retrieved. The exact number is recorded and logged by the variable `fails`. There are two reasons for failed lookups: 
1. The entity does not exist anymore, usually because it was created shortly before it was retrieved and deleted shortly afterwards because it was deemed irrelevant by moderators.
2. An UnknownSiteError occurred when trying to rerieve sitelinks for an entity. I assume this is the case when the entity has a Wikipedia page in a language that is not recorded in https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all. For example, when retrieving sitelinks for entity Q8873, this exception was thrown because language "bew" does not exist. bew is the language code for Basa Sunda (Sudanese), but this language code is missing from https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all. As a result, writers such as https://www.wikidata.org/wiki/Q484141 and https://www.wikidata.org/wiki/Q8873 are missing from our final dataset even though there are Wikipedia pages in all our target languages. 

In [17]:
import pywikibot
from pathlib import Path
import re
import polars as pl
import json
import logging
import sys

In [18]:
logging.basicConfig(
    level=logging.DEBUG, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(filename='property_retrieval.log'),
        logging.StreamHandler(sys.stdout)
    ]
)

In [5]:
def is_subset(labels, namespaces, special_labels):
    # Ensure at least one special label is present in namespaces (for Arabic: ar and arz)
    if not any(label in namespaces for label in special_labels):
        return False
    
    # Ensure for each label in labels, namespaces contains either the exact label or label followed by '-'
    for label in labels:
        if not any(ns == label or ns.startswith(f"{label}-") for ns in namespaces):
            return False
    
    return True

In [6]:
input_dir = Path("input")
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
labels = {"en", "de", "fr", "ru", "zh", "it", "pl", "hi"} 
arabic_labels = {'ar', 'arz'}
site = pywikibot.Site("wikidata", "wikidata")
repo = site.data_repository()
fails = 0

In [ ]:
# Process each input file individually
for file in input_dir.glob("*.csv"):
    logging.info(f"Processing file: {file}")
    
    # Load input TSV file and remove duplicate QIDs (input contains duplicates)
    query = pl.scan_csv(file).with_columns(
        pl.col("creator_id").alias("creator_id")
    ).unique(subset=["creator_id"])

    # Convert polars LazyFrame into DataFrame, select column "creator"
    entities_df = query.collect()
    q_numbers = entities_df.select("creator_id")

    # Create entities_dict with keys for 'creator' and 'num_sitelinks'
    entities_dict = { 
        "creator_id": [],
        "num_sitelinks": [],
    }

    # Process each QID individually
    for q_number in q_numbers.iter_rows():
        item = pywikibot.ItemPage(repo, q_number[0])
        if not item.exists():
            logging.warning(f"{q_number} does not exist")
            fails += 1
            continue
        
        logging.info(f"Processing Q-number: {q_number}")
        
        try: 
            logging.debug(f"Retrieving sitelinks for {q_number}")
            sitelinks = list(item.iterlinks("wikipedia"))  
            num_sitelinks = len(sitelinks)  
            namespaces = {re.search(r"\[\[(.+?):", str(sitelink)).group(1) for sitelink in sitelinks}
        except (pywikibot.exceptions.UnknownSiteError, pywikibot.exceptions.IsRedirectPageError) as e:
            logging.warning(f"Error processing {q_number}: {e}", exc_info=True)
            fails += 1 
            continue

        # Check if item has at least as many sitelinks as our chosen languages, and if yes, check if sitelinks match our languages
        if num_sitelinks > len(labels) and is_subset(labels, namespaces, arabic_labels): 
            logging.debug(f"Labels are a subset of {namespaces}")

            # Append QID, number of sitelinks, labels for entity to the dictionary
            entities_dict["creator_id"].append(q_number[0])
            entities_dict["num_sitelinks"].append(num_sitelinks)


    # Write entities_dict for current input file to JSON file
    output_file = output_dir / file.with_suffix('.json').name
    logging.info(f"Writing output to {output_file}")
    
    with output_file.open('w', encoding='utf-8') as f_out:
        json.dump(entities_dict, f_out, ensure_ascii=False, indent=4)
    
    logging.info(f"Data for {file} written successfully")
    logging.info(f"Failed lookups: {fails}")

## Merge Output JSON Files with Sitelink Counts and JSON Files with Properties 

Input: 
1. JSON files containing entity QIDs and number of sitelinks for all entities with at least as many sitelinks as input languages.
2. JSON files containing values for properties for all QIDs
  
Output: Merged JSON files containing sitelink count and properties 

Input files:
1. Sitelinks: https://drive.google.com/drive/folders/13VTPzeKkonprQzXTfuDrVE2hw6SBqC6C
2. Properties: https://drive.google.com/drive/folders/1Xc1yMFzYhlU5F5exmO8C-4LpjNo6kMGK

Output files: https://drive.google.com/drive/folders/1MfYOxqukqKWFPpYNaCr5dZV9IbGLsvjp


**Notes:**

The merged JSON files for Arabic, Italian and French are missing 3 (in the case of Arabic) and 1 (in the case of Italian and French) entities that were present in the input CSV files and in the output JSON with the sitelink counts. These entities were missing from the JSON files, i.e. they got lost when merging the output JSON files with Moulouds JSON files. One of the missing entities for Arabic is Q467895, if you want to check yourself.

In [2]:
from pathlib import Path
import json

In [8]:
input_1_dir = Path("dataset_v1_json")
input_2_dir = Path("output")
output_dir = Path("merged_results")
output_dir.mkdir(parents=True, exist_ok=True)
input_1_files = {file.stem: file for file in input_1_dir.glob("*.json")}
input_2_files = {file.stem.replace('filtered_entities_ids_', ''): file for file in input_2_dir.glob("*.json")}

In [ ]:
# Process input files that share the same language in the filename simultaneously
for language in input_1_files.keys() & input_2_files.keys():
    logging.info(f"Processing language: {language}")

    # Load input JSON files
    with open(input_1_files[language], 'r') as f1, open(input_2_files[language], 'r') as f2:
        properties_dict = json.load(f1)
        sitelinks_dict = json.load(f2)
    
    # Create merged dictionary with properties and sitelink counts
    merged_dict = {}
    for q_number, num_sitelinks in zip(sitelinks_dict["creator_id"], sitelinks_dict["num_sitelinks"]):
        if q_number in properties_dict:  
            merged_dict[q_number] = properties_dict[q_number].copy()  
            merged_dict[q_number]["num_sitelinks"] = num_sitelinks  

    # Write merged data back to JSON file
    output_file = output_dir / f"{language}_merged.json"
    logging.info(f"Writing output to {output_file}")
    
    with output_file.open('w', encoding='utf-8') as f_out:
        json.dump(merged_dict, f_out, ensure_ascii=False, indent=4)
    

## ToDos 

* Modify the structure of the output JSON file containing sitelink counts to match the structure of the JSON files containing properties; replace zip() in merge logic.
* Find a way to record sitelinks even when UnknownSiteError is thrown to ensure affected entities are not lost. 